# XGBoost (Alternative - Could be used)

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_squared_error
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder

print("--- XGBoost Classifier / Regressor ---")

# Classification
df_cls = pd.read_csv(r"../data/Crop_recommendation.csv")
df_cls.columns = [c.strip().lower() for c in df_cls.columns]
if 'label' in df_cls.columns:
    df_cls = df_cls.rename(columns={'label': 'crop_type'})
df_cls = df_cls.dropna(subset=['n', 'p', 'k', 'temperature', 'humidity', 'rainfall', 'crop_type'])

X_cls = df_cls[['n', 'p', 'k', 'temperature', 'humidity', 'rainfall']]
y_cls = df_cls['crop_type']

# XGBoost requires target labels to be numeric
le = LabelEncoder()
y_cls_encoded = le.fit_transform(y_cls)

X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(X_cls, y_cls_encoded, test_size=0.2, random_state=42)

clf = xgb.XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=42)
clf.fit(X_train_cls, y_train_cls)
y_pred_cls = clf.predict(X_test_cls)
print(f"XGBoost Classification Accuracy: {accuracy_score(y_test_cls, y_pred_cls):.4f}")

# Regression
try:
    df_reg = pd.read_csv(r"../data/master_dataset.csv")
    df_reg.columns = [c.strip().lower() for c in df_reg.columns]
    
    feature_cols = [c for c in ['temperature', 'rainfall', 'humidity', 'n', 'p', 'k'] if c in df_reg.columns]
    df_reg = df_reg.dropna(subset=['crop_yield'] + feature_cols)
    
    X_reg = df_reg[feature_cols]
    y_reg = pd.to_numeric(df_reg['crop_yield'], errors='coerce')
    
    X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)
    
    reg = xgb.XGBRegressor(objective='reg:squarederror', random_state=42)
    reg.fit(X_train_reg, y_train_reg)
    y_pred_reg = reg.predict(X_test_reg)
    print(f"XGBoost Regression RMSE: {mean_squared_error(y_test_reg, y_pred_reg) ** 0.5:.4f}")
except FileNotFoundError:
    print("master_dataset.csv not found for Regression testing.")



ModuleNotFoundError: No module named 'xgboost'

### Comment
XGBoost could be used and often yields slightly higher accuracy than Random Forest. However, it was not used because it is more prone to overfitting on smaller agricultural datasets and requires intense hyperparameter tuning, making it slightly more brittle for a simple MLOps starter pipeline.